# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 22 · Goal-aligned relationship representation

**Round 10: two matched models, one existing fold.** Round 9 found substantial input differences but little prediction response. Nonzero gradients and changed weights rule out a completely disconnected pathway, not other representation/optimization limitations. This notebook does not train. The new relationship interface is shared by both future arms; only six goal-frame numerical values differ. No prior best score is replaced.

In [ ]:
from pathlib import Path
import json, sys, subprocess, signal, os
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round10')
OUT = Path('/home/sagemaker-user/nfl-feature-round10-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Use the existing NFL space and upload/extract the Round 10 kit.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage, *options):
    child = subprocess.Popen([str(PY),str(KIT/'run_round.py'),stage,*options],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)
    try:
        for line in child.stdout:
            print(line, end='')
        code=child.wait()
    except KeyboardInterrupt:
        os.killpg(child.pid, signal.SIGINT)
        try: child.wait(timeout=8)
        except subprocess.TimeoutExpired:
            os.killpg(child.pid, signal.SIGTERM)
            try: child.wait(timeout=4)
            except subprocess.TimeoutExpired: os.killpg(child.pid, signal.SIGKILL);child.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve checkpoints and export the report. Do not change settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Completed audit evidence
These three figures are your uploaded real Round 9 aggregates. Prediction-change RMS is not RMSE, and zeroing interventions may be off-distribution.

In [ ]:
show(visuals.prior_inputs(KIT),'round9_input_differences')
show(visuals.prior_sensitivity(KIT),'round9_prediction_sensitivity')
show(visuals.prior_weights(KIT),'round9_weight_movement')

## Verify the immutable parent chain
Require `grouped_preflight_passed`. This checks bytes; it is not numerical replay of the old models. It does not launch compute or modify GitHub.

In [ ]:
run('preflight')

## 32-training-play feature smoke
Require `grouped_smoke_passed`. Reconstruct the six unchanged Round 9 formulas and verify the same support population. No target arrays are unpacked.

In [ ]:
run('smoke')

## Prepare only the extension
Proceed only after smoke passes. Reuse all 851 parent tensors, the existing split and targets. Require `grouped_features_ready`; new files contain goal arrays/masks/groups only.

In [ ]:
run('prepare')
show(visuals.support(OUT),'goal_channel_support')
show(visuals.magnitude(OUT),'goal_channel_magnitude')

## Stop point
Save this notebook. Continue to notebook 23 only when preparation passed. Do not edit group definitions, feature scales, model settings or bins after inspecting results. No new predictive score has been obtained yet.